# Lab 4 — Reading and Writing Data Files

**Course:** Python for AI & Data  
**Analyst:** *Janet Beaton*  
**Date:** *14 April 2026*

---

## The BeanScene Scenario

Sarah has exported BeanScene's menu data to a CSV file and stored her performance thresholds in a JSON config file. Your job is to load both files, run the menu analysis, and save the results back to disk.

Work from top to bottom. Run every cell as you go.


---
## Task 1 — Confirm Your Environment

Before opening any files, confirm:
1. Your working directory is the repo root (not inside `notebooks/`)
2. The data files exist at the expected paths

If the paths show `False`, close JupyterLab, `cd` to the repo root, and relaunch.


In [36]:
import os
from pathlib import Path

# Confirm working directory
print("Working directory:", os.getcwd())

# Define paths
RAW_DIR = Path("../data") / "raw"
PROCESSED_DIR = Path("../data") / "processed"

menu_path = RAW_DIR / "beanscene_menu.csv"
config_path = RAW_DIR / "beanscene_config.json"

# Verify files exist
print("Menu CSV exists:", menu_path.exists())
print("Config JSON exists:", config_path.exists())


Working directory: C:\Users\Jbeat\Homework\lab-04-reading-and-writing-data-files\notebooks
Menu CSV exists: True
Config JSON exists: True


---
## Task 2 — Load the Menu from CSV

Read `beanscene_menu.csv` into a list of dictionaries.

After loading:
- Convert `price` to `float` and `units_sold` to `int` for each row
- `name` and `category` should remain strings

Print the first item and check the types to confirm everything loaded correctly.

**Hint:** Use `csv.DictReader` — it reads each row as a dictionary using the header row as keys.


In [37]:
import csv

menu = []

# Open the CSV file and read into menu (list of dicts with correct types)
with open(menu_path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    # print(menu)
    for row in reader:
        # print(row)
        menu.append({
            "name": row["name"],
            "price": float(row["price"]),
            "units_sold": int(row["units_sold"]),
            "category": row["category"]
    })
# Print the first item and verify types
print(menu[0])
print("price type:", type(menu[0]["price"]))
print("units_sold type:", type(menu[0]["units_sold"]))
print(f"Total items loaded: {len(menu)}")


{'name': 'Espresso', 'price': 3.5, 'units_sold': 42, 'category': 'Hot Drinks'}
price type: <class 'float'>
units_sold type: <class 'int'>
Total items loaded: 8


---
## Task 3 — Load the Config from JSON

Read `beanscene_config.json` into a Python dictionary.

After loading, print:
- The café name
- The `high` performance threshold
- The `low_performer_threshold`

**Note:** Unlike CSV, JSON numbers load as actual Python numbers — no conversion needed.


In [38]:
import json

config = {}
with open(config_path, encoding="utf-8") as f:
    config = json.load(f)
# Open the JSON file and load into config


# Print key values from the config
print("Café name:", config["cafe_name"])
print("High threshold:", config["performance_thresholds"]["high"])
print("Low threshold:", config["low_performer_threshold"])


Café name: BeanScene
High threshold: 250.0
Low threshold: 150.0


---
## Task 4 — Analyse the Menu

Using the loaded `menu` list and `config` dictionary:

1. Calculate revenue for each item (`price * units_sold`)
2. Classify each item as `"High"`, `"Medium"`, or `"Low"` using the thresholds from `config` — **not hardcoded numbers**
3. Store the results in a new list called `results` — each item should be a dict with keys: `name`, `revenue`, `tier`
4. Print the results in a readable format

**Tip:** You can reuse the `classify_item` logic from Lab 3 — either copy the function or rewrite it here.


In [40]:
# Extract thresholds from config
high_threshold = config["performance_thresholds"]["high"]
low_threshold = config["low_performer_threshold"]

results = []
for item in menu:
    revenue=item["price"] * item["units_sold"]
    # print(f"{item['name']:<14}: ${revenue:.2f}")
# Loop over menu, calculate revenue, classify each item, append to results
    if revenue >= high_threshold:
        tier = "High"
    elif revenue >= low_threshold:
        tier = "Medium"
    else:
        tier = "Low"
    print(f"{item['name']}: ${revenue:.2f} [{tier}]")
    results.append({
        "name": item['name'],
        "revenue": revenue,
        "tier": tier})
# print(results)    

Espresso: $147.00 [Low]
Latte: $289.75 [High]
Cappuccino: $171.00 [Medium]
Cold Brew: $145.00 [Low]
Flat White: $199.75 [Medium]
Matcha Latte: $181.50 [Medium]
Iced Americano: $220.00 [Medium]
Chai Latte: $194.75 [Medium]


---
## Task 5 — Save Results to CSV

Write your `results` list to `data/processed/menu_results.csv`.

The output file should have three columns: `name`, `revenue`, `tier`.

Make sure `data/processed/` exists before writing (create it if needed).


In [41]:
# Create the processed directory if it doesn't exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
menu_results=results
# print(menu_results)
# Write results to CSV
with open("../data/processed/menu_results.csv", mode='w', newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'revenue', 'tier'])
    writer.writeheader()
    writer.writerows(results)
    
output_csv = PROCESSED_DIR / "menu_results.csv"
print("Saved:", output_csv)


Saved: ..\data\processed\menu_results.csv


---
## Task 6 — Save a Summary to JSON

Build a summary dictionary and write it to `data/processed/weekly_summary.json`.

Include at minimum:
- `cafe_name` (from config)
- `week` (from config: `analysis_week`)
- `total_revenue` (sum of all revenues)
- `items_analysed` (number of items)
- `best_seller` (name of the item with highest revenue)

Use `indent=2` when writing so the file is human-readable.


In [42]:
# revenue_highest=0
# for item in results:
#     revenue=item["revenue"]
# # # Loop over menu, calculate revenue, classify each item, append to results
#     if revenue >= revenue_highest:
#         name_highest=item["name"]
#         revenue_highest=revenue

In [43]:
output_json = PROCESSED_DIR / "weekly_summary.json"
total_revenue=0
revenue_highest=0
with open(output_json, mode="w", encoding='utf-8') as f:
    for item in results:
        revenue=item["revenue"]
        total_revenue=revenue+total_revenue
        if revenue >= revenue_highest:
            name_highest=item["name"]
            print(name_highest)
            revenue_highest=revenue
        
    summary={"cafe_name": config["cafe_name"],
        "analysis_week": config["analysis_week"],
        "total_revenue": total_revenue,
        "items_analysed": len(menu),
        "best_seller":name_highest
        }
    json.dump(summary, f, indent=2)
print("Saved:", output_json)

Espresso
Latte
Saved: ..\data\processed\weekly_summary.json


---
## Task 7 — Verify Your Outputs

Re-load both files you just wrote and print their contents to confirm they were saved correctly.

This is a professional habit: always verify your outputs — don't assume they're correct.


In [35]:
import os
import json
import csv
from pathlib import Path
RAW_DIR = Path("../data") / "raw"
PROCESSED_DIR = Path("../data") / "processed"
# Re-load menu_results.csv and print all rows
menu_path=PROCESSED_DIR/'menu_results.csv'
with open(menu_path, newline="", encoding="utf-8") as f:
    menu = [*csv.DictReader(f)]
print("=== menu_results.csv ===")
for item in menu:
    print(f"{item['name']}, ${item['revenue']}, tier: {item['tier']}")
# print(f"{item['name']}: ${revenue:.2f} [{menu[tier]}]")

# Re-load weekly_summary.json and print
output_json = PROCESSED_DIR / "weekly_summary.json"
with open(output_json, encoding='utf-8') as f:
    confirm_json = json.load(f)
print("\n=== weekly_summary.json ===")
print(f"cafe: {confirm_json['cafe_name']} \nweek: {confirm_json['analysis_week']}, \ntotal revenue: {confirm_json['total_revenue']}")
print(f"number of items: {confirm_json['items_analysed']} \nbest seller: {confirm_json['best_seller']}")

=== menu_results.csv ===
Espresso, $147.0, tier: Low
Latte, $289.75, tier: High
Cappuccino, $171.0, tier: Medium
Cold Brew, $145.0, tier: Low
Flat White, $199.75, tier: Medium
Matcha Latte, $181.5, tier: Medium
Iced Americano, $220.0, tier: Medium
Chai Latte, $194.75, tier: Medium

=== weekly_summary.json ===
cafe: BeanScene 
week: 2024-W04, 
total revenue: 1548.75
number of items: 8 
best seller: Latte


---
## ⭐ Optional Advanced Tasks

### Optional 1 — Low Performers File
Write a separate CSV to `data/processed/low_performers.csv` containing only the items classified as `"Low"`. Then look up append mode (`"a"`) and add a final row with the count of low performers.

### Optional 2 — Config-Driven Thresholds
Confirm your analysis is fully config-driven: open `data/raw/beanscene_config.json`, change the `high` threshold from `250` to `200`, save the file, and re-run Task 4. Verify the tiers change without editing any Python code. Then change it back.

### Optional 3 — Category Summary
Group the menu items by `category` and write a file `data/processed/category_summary.json` showing total revenue per category. (Hint: use a dictionary to accumulate revenue per category as you loop.)


In [31]:
# Optional tasks — work here
